# Requiem — Pipeline organizado do TCC

Este notebook reúne, **na ordem correta de execução**, os módulos do projeto.

Cada célula `%%writefile` grava o respectivo módulo `.py` em disco.

**Etapas:**
1. Instalar dependências
2. Módulos base (sem dependências locais)
3. Módulos de alto nível (motor de harmonia)
4. Geração do dataset (`music_data/`)
5. **Treinamento do modelo → gera os pesos** (`saved_model/model.keras`)
6. Exportação dos pesos (`weights.json` / TF.js)
7. Validação e testes


## ⚠️ Revisão: correções de qualidade da harmonia gerada

1. **`requiem_model.py` agora usa a LSTM.** O motor neural é ligado através de `RequiemModel(..., use_neural=True)`.
2. **Vazamento de dados corrigido.** O split de treino/validação é feito por **música inteira**.
3. **Data augmentation otimizado.** Treino concentrado nas progressões já normalizadas.
4. **Balanceamento de escala.** `acoustic_score` normalizado pela densidade de notas.

## 1. Instalar dependências

In [1]:
!pip install -q tensorflow numpy pytest tensorflowjs

## 2. Módulos base (sem dependências locais)

In [2]:
%%writefile tonality_adapter.py
from __future__ import annotations
import re
from dataclasses import dataclass
from typing import Optional

@dataclass
class DetectedNote:
    pitch: float
    start_time: float
    end_time: float
    amplitude: float = 0.7

ROOT_OFFSETS: dict[str, int] = {
    "C": 0,  "C#": 1,  "Db": 1,
    "D": 2,  "D#": 3,  "Eb": 3,
    "E": 4,  "Fb": 4,
    "F": 5,  "F#": 6,  "Gb": 6,
    "G": 7,  "G#": 8,  "Ab": 8,
    "A": 9,  "A#": 10, "Bb": 10,
    "B": 11, "Cb": 11,
}

TONALITY_OFFSETS: dict[str, int] = {
    "C": 0,   "C#": 1,  "Db": 1,
    "D": 2,   "D#": 3,  "Eb": 3,
    "E": 4,
    "F": 5,   "F#": 6,  "Gb": 6,
    "G": 7,   "G#": 8,  "Ab": 8,
    "A": 9,   "A#": 10, "Bb": 10,
    "B": 11,  "Cb": 11,
    "Am": 0,   "A#m": 1,  "Bbm": 1,
    "Bm": 2,   "Cm": 3,
    "C#m": 4,  "Dm": 5,
    "D#m": 6,  "Ebm": 6,  "Em": 7,
    "Fm": 8,   "F#m": 9,
    "Gm": 10,  "G#m": 11, "Abm": 11,
}

NOTE_NAMES_SHARP = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
NOTE_NAMES_FLAT  = ["C", "Db", "D", "Eb", "E", "F", "Gb", "G", "Ab", "A", "Bb", "B"]

FLAT_TONALITIES: set[str] = {
    "F", "Bb", "Eb", "Ab", "Db", "Gb", "Cb",
    "Dm", "Gm", "Cm", "Fm", "Bbm", "Ebm", "Abm",
}

CHORD_REGEX = re.compile(r"^([A-G][#b]?)(.*)$")
INTERVAL_TO_ROMAN = ["I", "bII", "II", "bIII", "III", "IV", "bV", "V", "bVI", "VI", "bVII", "VII"]

SUFFIX_DICT: dict[str, list[int]] = {
    "":        [0, 4, 7],
    "m":       [0, 3, 7],
    "dim":     [0, 3, 6],
    "aug":     [0, 4, 8],
    "+":       [0, 4, 8],
    "sus4":    [0, 5, 7],
    "sus2":    [0, 2, 7],
    "5":       [0, 7],
    "maj7":    [0, 4, 7, 11],
    "7":       [0, 4, 7, 10],
    "m7":      [0, 3, 7, 10],
    "m7b5":    [0, 3, 6, 10],
    "dim7":    [0, 3, 6, 9],
    "add9":    [0, 4, 7, 2],
    "madd9":   [0, 3, 7, 2],
    "sus4add9":[0, 5, 7, 2],
    "11":      [0, 4, 7, 10, 2, 5],
    "m11":     [0, 3, 7, 10, 2, 5],
}

MAJOR_TONALITIES = [{"value": "C", "label": "Dó Maior (C)"}]
MINOR_TONALITIES = [{"value": "Am", "label": "Lá Menor (Am)"}]
ALL_TONALITIES = MAJOR_TONALITIES + MINOR_TONALITIES
UT_OPTIONS = [2, 4, 8]
QT_OPTIONS = [2, 3, 4, 6, 9, 12]

def _parse_chord(chord_name: str) -> Optional[tuple[str, str]]:
    m = CHORD_REGEX.match(chord_name)
    if not m: return None
    return m.group(1), m.group(2)

def _note_names_for(tonality: str) -> list[str]:
    return NOTE_NAMES_FLAT if tonality in FLAT_TONALITIES else NOTE_NAMES_SHARP

def get_tonality_offset(tonality: str) -> int:
    return TONALITY_OFFSETS.get(tonality, 0)

def normalize_notes(notes: list[DetectedNote], target_tonality: str) -> list[DetectedNote]:
    offset = get_tonality_offset(target_tonality)
    if offset == 0:
        return [DetectedNote(n.pitch, n.start_time, n.end_time, n.amplitude) for n in notes]
    return [DetectedNote(n.pitch - offset, n.start_time, n.end_time, n.amplitude) for n in notes]

def transpose_chord(chord_name: str, target_tonality: str) -> str:
    parsed = _parse_chord(chord_name)
    if not parsed: return chord_name
    root, suffix = parsed
    offset = get_tonality_offset(target_tonality)
    if offset == 0: return chord_name
    root_index = ROOT_OFFSETS.get(root)
    if root_index is None: return chord_name
    new_index = (root_index + offset) % 12
    note_names = _note_names_for(target_tonality)
    return note_names[new_index] + suffix

def transpose_progression(progression: list[str], target_tonality: str) -> list[str]:
    return [transpose_chord(c, target_tonality) for c in progression]

def normalize_progression_to_c(progression: list[str], original_tonality: str) -> list[str]:
    offset = get_tonality_offset(original_tonality)
    if offset == 0: return list(progression)
    result = []
    for chord_name in progression:
        parsed = _parse_chord(chord_name)
        if not parsed:
            result.append(chord_name)
            continue
        root, suffix = parsed
        root_index = ROOT_OFFSETS.get(root)
        if root_index is None:
            result.append(chord_name)
            continue
        new_index = ((root_index - offset) % 12 + 12) % 12
        result.append(NOTE_NAMES_SHARP[new_index] + suffix)
    return result

def chord_to_roman(chord: str, tonality: str) -> str:
    parsed = _parse_chord(chord)
    if not parsed: return chord
    root, suffix = parsed
    root_offset = ROOT_OFFSETS.get(root)
    if root_offset is None: return chord
    tonality_offset = get_tonality_offset(tonality)
    interval = (root_offset - tonality_offset + 12) % 12
    roman = INTERVAL_TO_ROMAN[interval]
    is_minor = suffix.startswith("m") and not suffix.startswith("maj")
    is_dim = suffix.startswith("dim")
    if is_minor or is_dim: roman = roman.lower()
    if suffix == "m": suffix = ""
    elif suffix.startswith("m") and not suffix.startswith("maj"): suffix = suffix[1:]
    return roman + suffix

ROMAN_REGEX = re.compile(r"^([b#]?(?:III|iii|II|ii|IV|iv|VIII|viii|VII|vii|VI|vi|V|v|I|i))((?:.*))$")

def roman_to_chord(roman: str, tonality: str) -> str:
    m = ROMAN_REGEX.match(roman)
    if not m: return roman
    base_roman = m.group(1)
    suffix = m.group(2)
    search_base = base_roman.upper()
    if search_base.startswith("B"): search_base = "b" + search_base[1:]
    if search_base.startswith("#"): search_base = "#" + search_base[1:]
    try: interval = INTERVAL_TO_ROMAN.index(search_base)
    except ValueError: return roman
    is_lower = base_roman == base_roman.lower()
    if is_lower and not suffix.startswith("dim"): suffix = "m" + suffix
    tonality_offset = get_tonality_offset(tonality)
    root_index = (tonality_offset + interval) % 12
    note_names = _note_names_for(tonality)
    return note_names[root_index] + suffix

def get_chord_pitch_classes(chord_name: str) -> list[int]:
    parsed = _parse_chord(chord_name)
    if not parsed: return [0, 4, 7]
    root, suffix = parsed
    root_index = ROOT_OFFSETS.get(root, 0)
    intervals = SUFFIX_DICT.get(suffix)
    if intervals is None:
        if suffix.startswith("m") and not suffix.startswith("maj"): intervals = [0, 3, 7]
        elif suffix.startswith("sus"): intervals = [0, 5, 7]
        else: intervals = [0, 4, 7]
    return [(root_index + i) % 12 for i in intervals]


Overwriting tonality_adapter.py


In [3]:
%%writefile audio_analyzer.py
from __future__ import annotations
import math
from tonality_adapter import DetectedNote

MAJOR_PROFILE = [6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88]
MINOR_PROFILE = [6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17]
PITCH_CLASS_NAMES_MAJOR = ["C","Db","D","Eb","E","F","F#","G","Ab","A","Bb","B"]
PITCH_CLASS_NAMES_MINOR = ["C","C#","D","D#","E","F","F#","G","G#","A","Bb","B"]

def detect_key(notes: list[DetectedNote]) -> str:
    if not notes or len(notes) < 3: return "C"
    durations = [0.0] * 12
    for note in notes:
        if note.pitch < 0: continue
        dur = note.end_time - note.start_time
        pc = int(note.pitch) % 12
        durations[pc] += dur
    total = sum(durations)
    if total == 0: return "C"
    normalized = [d / total for d in durations]

    def pearson(inp: list[float], profile: list[float]) -> float:
        avg_inp = sum(inp) / 12
        avg_prof = sum(profile) / 12
        num = sum((inp[i] - avg_inp) * (profile[i] - avg_prof) for i in range(12))
        den1 = sum((inp[i] - avg_inp) ** 2 for i in range(12))
        den2 = sum((profile[i] - avg_prof) ** 2 for i in range(12))
        denom = math.sqrt(den1 * den2)
        return num / denom if denom != 0 else 0.0

    best_key = "C"
    max_corr = -math.inf
    for shift in range(12):
        shifted = normalized[shift:] + normalized[:shift]
        corr_major = pearson(shifted, MAJOR_PROFILE)
        if corr_major > max_corr:
            max_corr = corr_major
            best_key = PITCH_CLASS_NAMES_MAJOR[shift]
        corr_minor = pearson(shifted, MINOR_PROFILE)
        if corr_minor > max_corr:
            max_corr = corr_minor
            best_key = f"{PITCH_CLASS_NAMES_MINOR[shift]}m"
    return best_key

def estimate_bpm(notes: list[DetectedNote]) -> int:
    if not notes or len(notes) < 2: return 120
    sorted_notes = sorted(notes, key=lambda n: n.start_time)
    deltas: list[float] = []
    for i in range(1, len(sorted_notes)):
        delta = sorted_notes[i].start_time - sorted_notes[i - 1].start_time
        if 0.1 < delta < 2.0: deltas.append(delta)
    if not deltas: return 120
    deltas.sort()
    if len(deltas) > 5:
        trim = max(1, int(len(deltas) * 0.1))
        deltas = deltas[trim: len(deltas) - trim]
    avg_delta = sum(deltas) / len(deltas)
    bpm = round(60 / avg_delta)
    while bpm < 60: bpm *= 2
    while bpm > 180: bpm = bpm // 2
    return max(60, min(180, bpm))


Overwriting audio_analyzer.py


In [4]:
%%writefile harmony_engine.py
from __future__ import annotations
import re
import json
import math
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
from tonality_adapter import DetectedNote, chord_to_roman, get_chord_pitch_classes, TONALITY_OFFSETS

@dataclass
class ChordNode:
    notes: list[int]
    allowed_transitions: list[str]

@dataclass
class ProgressionData:
    title: str
    original_tonality: str
    normalized_progression: list[str]

TransitionMatrix = dict[str, dict[str, float]]

@dataclass
class TwoTierMarkovModel:
    base_matrix: TransitionMatrix = field(default_factory=dict)
    suffix_matrix: TransitionMatrix = field(default_factory=dict)

@dataclass
class ChordResult:
    chord: str
    velocity: float

HARMONY_GRAPH: dict[str, ChordNode] = {
    "C":   ChordNode([0,4,7],   ["C","G","F","Am","Dm","Em","G7","C7","E7","A7","D7","Bdim"]),
    "G":   ChordNode([7,11,2],  ["G","C","D","Em","Am","Bm","D7","G7","B7","E7","F#dim"]),
    "D":   ChordNode([2,6,9],   ["D","G","A","Bm","Em","F#m","A7","D7","F#dim","E7"]),
    "A":   ChordNode([9,1,4],   ["A","D","E","F#m","Bm","C#m","E7","A7","C#dim","B7"]),
    "E":   ChordNode([4,8,11],  ["E","A","B","C#m","F#m","G#m","B7","E7","G#dim"]),
    "B":   ChordNode([11,3,6],  ["B","E","F#","G#m","C#m","D#m","F#7","B7"]),
    "F#":  ChordNode([6,10,1],  ["F#","B","C#","D#m","G#m","F#7","C#7"]),
    "F":   ChordNode([5,9,0],   ["F","C","Bb","Dm","Am","Gm","C7","F7","D7","A7"]),
    "Bb":  ChordNode([10,2,5],  ["Bb","F","Eb","Gm","Dm","Cm","F7","Bb7","D7"]),
    "Eb":  ChordNode([3,7,10],  ["Eb","Bb","Ab","Cm","Gm","Fm","Bb7","Eb7","G7"]),
    "Ab":  ChordNode([8,0,3],   ["Ab","Eb","Db","Fm","Cm","Bbm","Eb7","Ab7"]),
    "Db":  ChordNode([1,5,8],   ["Db","Ab","Gb","Bbm","Fm","Ab7","Db7"]),
    "Am":  ChordNode([9,0,4],   ["Am","C","Dm","Em","F","G","E7","G7","A7","Bdim","D7"]),
    "Em":  ChordNode([4,7,11],  ["Em","G","Am","C","D","Bm","B7","D7","F#dim","E7"]),
    "Bm":  ChordNode([11,2,6],  ["Bm","D","Em","G","A","F#m","F#7","A7","F#dim"]),
    "F#m": ChordNode([6,9,1],   ["F#m","A","Bm","D","E","C#m","C#7","E7","C#dim"]),
    "C#m": ChordNode([1,4,8],   ["C#m","E","F#m","A","B","G#m","G#7","B7","G#dim"]),
    "G#m": ChordNode([8,11,3],  ["G#m","B","C#m","E","F#","D#m","D#7","F#7"]),
    "D#m": ChordNode([3,6,10],  ["D#m","F#","G#m","B","D#7","A#7"]),
    "Dm":  ChordNode([2,5,9],   ["Dm","F","Am","C","Bb","Gm","A7","C7","D7","Bdim"]),
    "Gm":  ChordNode([7,10,2],  ["Gm","Bb","Dm","F","Eb","Cm","D7","F7","Bb7"]),
    "Cm":  ChordNode([0,3,7],   ["Cm","Eb","Gm","Bb","Ab","Fm","G7","Bb7","Eb7"]),
    "Fm":  ChordNode([5,8,0],   ["Fm","Ab","Cm","Eb","Db","Bbm","C7","Eb7","Ab7"]),
    "Bbm": ChordNode([10,1,5],  ["Bbm","Db","Fm","Ab","F7","Ab7","Db7"]),
    "G7":  ChordNode([7,11,2,5],  ["C","Am","Cm","G7","C7","F","Dm"]),
    "D7":  ChordNode([2,6,9,0],   ["G","Em","Gm","D7","G7","C","Am"]),
    "A7":  ChordNode([9,1,4,7],   ["D","Bm","Dm","A7","D7","G","Em"]),
    "E7":  ChordNode([4,8,11,2],  ["A","F#m","Am","E7","A7","D","Bm"]),
    "B7":  ChordNode([11,3,6,9],  ["E","C#m","Em","B7","E7","A","F#m"]),
    "F#7": ChordNode([6,10,1,4],  ["B","G#m","Bm","F#7","B7","E","C#m"]),
    "C#7": ChordNode([1,5,8,11],  ["F#","D#m","F#m","C#7","F#7","B"]),
    "C7":  ChordNode([0,4,7,10],  ["F","Dm","Fm","C7","F7","Bb","Gm"]),
    "F7":  ChordNode([5,9,0,3],   ["Bb","Gm","Bbm","F7","Bb7","Eb","Cm"]),
    "Bb7": ChordNode([10,2,5,8],  ["Eb","Cm","Bb7","Eb7","Ab","Fm"]),
    "Eb7": ChordNode([3,7,10,1],  ["Ab","Fm","Eb7","Ab7","Db"]),
    "Ab7": ChordNode([8,0,3,6],   ["Db","Bbm","Ab7","Db7"]),
    "Db7": ChordNode([1,5,8,11],  ["Gb","Db7","Ab","Fm"]),
    "D#7": ChordNode([3,7,10,1],  ["G#m","D#7","G#","F"]),
    "G#7": ChordNode([8,0,3,6],   ["C#m","G#7","C#","F#m"]),
    "A#7": ChordNode([10,2,5,8],  ["D#m","A#7","D#","G#m"]),
    "Bdim":  ChordNode([11,2,5], ["C","Am","G7","Dm","F","Em"]),
    "F#dim": ChordNode([6,9,0],  ["G","Em","D7","Am","C","Bm"]),
    "C#dim": ChordNode([1,4,7],  ["D","Bm","A7","Em","G","F#m"]),
    "G#dim": ChordNode([8,11,2], ["A","F#m","E7","Bm","D","C#m"]),
    "D#dim": ChordNode([3,6,9],  ["E","C#m","B7","F#m","A"]),
    "A#dim": ChordNode([10,1,4], ["B","G#m","F#7","C#m","E"]),
    "Edim":  ChordNode([4,7,10], ["F","Dm","C7","Am","Bb","Gm"]),
    "Adim":  ChordNode([9,0,3],  ["Bb","Gm","F7","Dm","Eb","Cm"]),
    "Ddim":  ChordNode([2,5,8],  ["Eb","Cm","Bb7","Gm","Ab","Fm"]),
    "Gdim":  ChordNode([7,10,1], ["Ab","Fm","Eb7","Cm","Db"]),
    "Cdim":  ChordNode([0,3,6],  ["Db","Bbm","Ab7","Fm"]),
    "Fdim":  ChordNode([5,8,11], ["Gb","Ebm","Db7","Bbm"]),
    "C#":  ChordNode([1,5,8],  ["Db","Ab","Gb","Bbm","Fm","Ab7","Db7"]),
    "Gb":  ChordNode([6,10,1], ["F#","B","C#","D#m","G#m","F#7","C#7"]),
    "G#":  ChordNode([8,0,3],  ["Ab","Eb","Db","Fm","Cm","Bbm","Eb7","Ab7"]),
    "D#":  ChordNode([3,7,10], ["Eb","Bb","Ab","Cm","Gm","Fm","Bb7","Eb7","G7"]),
    "Ebm": ChordNode([3,6,10], ["D#m","F#","G#m","B","D#7","A#7"]),
}

WEIGHT_IN_CHORD = 2.0
WEIGHT_ROOT_BONUS = 1.5
WEIGHT_OUT_PENALTY = 0.5
WEIGHT_STRONG_BEAT = 3.0
WEIGHT_WEAK_BEAT = 0.5
BEAT_MARGIN_SEC = 0.15
REGISTER_LOW_THRESHOLD = 48
REGISTER_HIGH_THRESHOLD = 72
REGISTER_PENALTY = 0.2
REGISTER_BONUS = 1.5
DENSITY_THRESHOLD = 8
ORNAMENT_PENALTY_MULTIPLIER = 0.2
LEAP_THRESHOLD = 7
TENSION_BONUS = 1.5
WEIGHT_MARKOV = 20.0
THRESHOLD_MODAL_INTERCHANGE = 0.05
ALPHA = 0.1

def load_music_data(data_dir: str | Path = "music_data") -> list[ProgressionData]:
    path = Path(data_dir)
    datasets: list[ProgressionData] = []
    for f in sorted(path.glob("*.json")):
        try:
            raw = json.loads(f.read_text(encoding="utf-8"))
            if isinstance(raw.get("normalizedProgression"), list) and isinstance(raw.get("originalTonality"), str):
                datasets.append(ProgressionData(
                    title=raw.get("title", f.stem),
                    original_tonality=raw["originalTonality"],
                    normalized_progression=raw["normalizedProgression"],
                ))
        except Exception:
            pass
    return datasets

HARMONY_GRAPH_ROMANS: set[str] = set()
_ROMAN_SPLIT_RE = re.compile(r"^([b#]?(?:III|iii|II|ii|IV|iv|VIII|viii|VII|vii|VI|vi|V|v|I|i))(.*)$")

def _init_romans():
    global HARMONY_GRAPH_ROMANS
    HARMONY_GRAPH_ROMANS = {chord_to_roman(c, "C") for c in HARMONY_GRAPH}
_init_romans()

def split_cinematic(roman: str) -> tuple[str, str]:
    if roman in HARMONY_GRAPH_ROMANS: return roman, "none"
    m = _ROMAN_SPLIT_RE.match(roman)
    if not m: return roman, "none"
    return m.group(1), m.group(2) or "none"

def build_two_tier_markov_model(datasets: list[ProgressionData]) -> TwoTierMarkovModel:
    all_base_degrees = list({chord_to_roman(c, "C") for c in HARMONY_GRAPH})
    base_counts: dict[str, dict[str, float]] = {}
    for mod in datasets:
        prog = [chord_to_roman(c, mod.original_tonality) for c in mod.normalized_progression]
        base_prog = [split_cinematic(r)[0] for r in prog]
        for i in range(len(base_prog) - 1):
            from1 = base_prog[i]
            to = base_prog[i + 1]
            base_counts.setdefault(from1, {})[to] = base_counts.get(from1, {}).get(to, 0) + 1
            if i >= 1:
                from2 = base_prog[i - 1]
                state2 = f"{from2},{from1}"
                base_counts.setdefault(state2, {})[to] = base_counts.get(state2, {}).get(to, 0) + 1

    base_matrix: TransitionMatrix = {}
    for state, destinations in base_counts.items():
        row: dict[str, float] = {}
        total = 0.0
        for deg in all_base_degrees:
            count = destinations.get(deg, 0) + ALPHA
            row[deg] = count
            total += count
        for deg in all_base_degrees: row[deg] /= total
        base_matrix[state] = row

    all_extensions: set[str] = {"none","add9","madd9","sus4","sus2","maj7","m7b5","aug","m7","7","11","m11","5"}
    suffix_counts: dict[str, dict[str, float]] = {}
    for mod in datasets:
        prog = [chord_to_roman(c, mod.original_tonality) for c in mod.normalized_progression]
        for r in prog:
            base, ext = split_cinematic(r)
            suffix_counts.setdefault(base, {})[ext] = suffix_counts.get(base, {}).get(ext, 0) + 1
            all_extensions.add(ext)

    extensions_list = list(all_extensions)
    suffix_matrix: TransitionMatrix = {}
    for base_state, extensions in suffix_counts.items():
        row: dict[str, float] = {}
        total = 0.0
        for ext in extensions_list:
            count = extensions.get(ext, 0) + ALPHA
            row[ext] = count
            total += count
        for ext in extensions_list: row[ext] /= total
        suffix_matrix[base_state] = row
    return TwoTierMarkovModel(base_matrix=base_matrix, suffix_matrix=suffix_matrix)

_model_cache: Optional[TwoTierMarkovModel] = None
def get_model(data_dir: str | Path = "music_data") -> TwoTierMarkovModel:
    global _model_cache
    if _model_cache is None:
        datasets = load_music_data(data_dir)
        _model_cache = build_two_tier_markov_model(datasets)
    return _model_cache

def _compute_note_score(chord_notes: list[int], played_notes: list[DetectedNote], window_start: float, seconds_per_beat: float, is_high_density: bool, avg_amplitude: float) -> float:
    score = 0.0
    root_note = chord_notes[0]
    out_penalty = WEIGHT_OUT_PENALTY * ORNAMENT_PENALTY_MULTIPLIER if is_high_density else WEIGHT_OUT_PENALTY
    for note in played_notes:
        pc = round(note.pitch) % 12
        is_chord_tone = pc in chord_notes
        is_root = pc == root_note
        rel_time = note.start_time - window_start
        beat_phase = (rel_time % seconds_per_beat) / seconds_per_beat if seconds_per_beat > 0 else 0
        is_strong = beat_phase < BEAT_MARGIN_SEC or beat_phase > 1 - BEAT_MARGIN_SEC
        rhythmic_weight = WEIGHT_STRONG_BEAT if is_strong else WEIGHT_WEAK_BEAT
        amp_weight = (note.amplitude or 0.7) / avg_amplitude
        register_mod = 1.0
        if note.pitch < REGISTER_LOW_THRESHOLD and is_root: register_mod = REGISTER_BONUS
        if note.pitch > REGISTER_HIGH_THRESHOLD and not is_chord_tone: register_mod = REGISTER_PENALTY
        base_val = (WEIGHT_IN_CHORD + (WEIGHT_ROOT_BONUS if is_root else 0)) if is_chord_tone else -out_penalty
        score += base_val * rhythmic_weight * amp_weight * register_mod
    return score

def _lookup_style_probability(matrix: TransitionMatrix, state1: str, state2: Optional[str], target: str) -> float:
    if state2 and matrix.get(state2, {}).get(target) is not None: return matrix[state2][target]
    return matrix.get(state1, {}).get(target, 0.0)

def determine_next_chord(chord_history: list[str], played_notes: list[DetectedNote], window_start: float, seconds_per_beat: float, model: Optional[TwoTierMarkovModel] = None) -> ChordResult:
    if model is None: model = get_model()
    current_chord = chord_history[-1] if chord_history else "C"
    if not played_notes: return ChordResult(chord=current_chord, velocity=0.7)
    current_base_roman, _ = split_cinematic(chord_to_roman(current_chord, "C"))
    current_base_name = next((c for c in HARMONY_GRAPH if chord_to_roman(c, "C") == current_base_roman), "C")
    current_node = HARMONY_GRAPH.get(current_base_name)
    if not current_node: return ChordResult(chord=current_chord, velocity=0.7)
    roman_history = [split_cinematic(chord_to_roman(c, "C"))[0] for c in chord_history]
    curr_roman = roman_history[-1] if roman_history else "I"
    prev_roman = roman_history[-2] if len(roman_history) > 1 else None
    state1 = curr_roman
    state2 = f"{prev_roman},{curr_roman}" if prev_roman else None
    density = len(played_notes)
    is_high_density = density > DENSITY_THRESHOLD
    sum_pitch = sum(n.pitch for n in played_notes)
    sum_amp = sum(n.amplitude or 0.7 for n in played_notes)
    avg_pitch = sum_pitch / density if density > 0 else 0
    avg_amplitude = max(0.2, sum_amp / density) if density > 0 else 0.7
    has_tension_leap = any(abs(played_notes[i].pitch - played_notes[i-1].pitch) >= LEAP_THRESHOLD for i in range(1, density))
    all_extensions = list(model.suffix_matrix.get(curr_roman, {"none": 1.0}).keys()) or ["none"]
    best_chord = current_chord
    best_score = -math.inf
    for candidate_base in HARMONY_GRAPH:
        candidate_roman = chord_to_roman(candidate_base, "C")
        base_roman, _ = split_cinematic(candidate_roman)
        base_style_prob = _lookup_style_probability(model.base_matrix, state1, state2, base_roman)
        is_allowed_graph = candidate_base in current_node.allowed_transitions
        is_allowed_markov = base_style_prob > THRESHOLD_MODAL_INTERCHANGE
        if not is_allowed_graph and not is_allowed_markov: continue
        if has_tension_leap and ("7" in candidate_base or "dim" in candidate_base): base_style_prob *= TENSION_BONUS
        is_triad = "7" not in candidate_base and "dim" not in candidate_base
        valid_extensions = all_extensions if is_triad else ["none"]
        for ext in valid_extensions:
            full_chord = candidate_base if ext == "none" else candidate_base + ext
            full_notes = get_chord_pitch_classes(full_chord)
            suffix_prob = model.suffix_matrix.get(base_roman, {}).get(ext, 1.0 / len(all_extensions))
            combined_prob = base_style_prob * suffix_prob
            note_score = _compute_note_score(full_notes, played_notes, window_start, seconds_per_beat, is_high_density, avg_amplitude)
            final_score = note_score + combined_prob * WEIGHT_MARKOV
            if final_score > best_score:
                best_score = final_score
                best_chord = full_chord
    return ChordResult(chord=best_chord, velocity=avg_amplitude)

def generate_progression(played_notes: list[DetectedNote], bpm: float, time_signature_numerator: int, time_signature_denominator: int, start_chord: str = "C", model: Optional[TwoTierMarkovModel] = None) -> list[ChordResult]:
    if model is None: model = get_model()
    if not played_notes: return [ChordResult(chord=start_chord, velocity=0.7)]
    seconds_per_beat = (4 / time_signature_denominator) * (60 / bpm)
    seconds_per_measure = seconds_per_beat * time_signature_numerator
    total_start = min(n.start_time for n in played_notes)
    total_end = max(n.end_time for n in played_notes)
    total_duration = total_end - total_start
    window_count = max(1, math.ceil(total_duration / seconds_per_measure))
    progression: list[ChordResult] = []
    history = [start_chord]
    for w in range(window_count):
        window_start = total_start + w * seconds_per_measure
        window_end = window_start + seconds_per_measure
        window_notes = [n for n in played_notes if n.start_time < window_end and n.end_time > window_start]
        clipped: list[DetectedNote] = [
            DetectedNote(pitch=n.pitch, start_time=max(n.start_time, window_start), end_time=min(n.end_time, window_end), amplitude=n.amplitude)
            for n in window_notes
        ]
        result = determine_next_chord(history, clipped, window_start, seconds_per_beat, model)
        progression.append(result)
        history.append(result.chord)
    return progression


Overwriting harmony_engine.py


In [5]:
%%writefile chord_dataset.py
from __future__ import annotations
import json
from pathlib import Path
from typing import NamedTuple
import numpy as np
from tonality_adapter import normalize_progression_to_c, transpose_chord, NOTE_NAMES_SHARP, ROOT_OFFSETS

WINDOW_SIZE = 4
PAD_TOKEN = "<PAD>"

def load_raw_progressions(data_dir: str | Path = "music_data") -> list[dict]:
    path = Path(data_dir)
    raw_data = []
    for f in sorted(path.glob("*.json")):
        try:
            raw = json.loads(f.read_text(encoding="utf-8"))
            if isinstance(raw.get("normalizedProgression"), list):
                raw_data.append(raw)
        except Exception:
            pass
    return raw_data

def get_canonical_progressions(raw_data: list[dict]) -> list[list[str]]:
    return [normalize_progression_to_c(item["normalizedProgression"], item["originalTonality"]) for item in raw_data]

class Vocabulary:
    def __init__(self, chords: list[str]) -> None:
        unique = [PAD_TOKEN] + sorted(set(chords))
        self.idx_to_chord: list[str] = unique
        self.chord_to_idx: dict[str, int] = {c: i for i, c in enumerate(unique)}
    def __len__(self) -> int:
        return len(self.idx_to_chord)
    def encode(self, chord: str) -> int:
        return self.chord_to_idx.get(chord, 0)
    def decode(self, idx: int) -> str:
        if 0 <= idx < len(self.idx_to_chord): return self.idx_to_chord[idx]
        return PAD_TOKEN
    def save(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps({"vocab": self.idx_to_chord}, indent=2), encoding="utf-8")
    @classmethod
    def load(cls, path: str | Path) -> "Vocabulary":
        data = json.loads(Path(path).read_text(encoding="utf-8"))
        vocab = cls.__new__(cls)
        vocab.idx_to_chord = data["vocab"]
        vocab.chord_to_idx = {c: i for i, c in enumerate(vocab.idx_to_chord)}
        return vocab

def build_vocabulary(progressions: list[list[str]]) -> Vocabulary:
    all_chords = [chord for prog in progressions for chord in prog]
    return Vocabulary(all_chords)

class TrainingData(NamedTuple):
    X: np.ndarray
    y: np.ndarray
    vocab: Vocabulary

def create_training_data(progressions: list[list[str]], vocab: Vocabulary, window_size: int = WINDOW_SIZE) -> TrainingData:
    X_list: list[list[int]] = []
    y_list: list[int] = []
    pad_idx = vocab.encode(PAD_TOKEN)
    for prog in progressions:
        encoded = [vocab.encode(chord) for chord in prog]
        for i in range(len(encoded)):
            start = i - window_size
            context = []
            for j in range(start, i):
                if j < 0: context.append(pad_idx)
                else: context.append(encoded[j])
            target = encoded[i]
            X_list.append(context)
            y_list.append(target)
    X = np.array(X_list, dtype=np.int32)
    y = np.array(y_list, dtype=np.int32)
    return TrainingData(X=X, y=y, vocab=vocab)

class SplitTrainingData(NamedTuple):
    X_train: np.ndarray
    y_train: np.ndarray
    X_val: np.ndarray
    y_val: np.ndarray
    vocab: Vocabulary

def prepare_train_val_split(data_dir: str | Path = "music_data", window_size: int = WINDOW_SIZE, val_fraction: float = 0.2, seed: int = 42, verbose: bool = True) -> SplitTrainingData:
    raw = load_raw_progressions(data_dir)
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(raw))
    n_val = max(1, round(len(raw) * val_fraction))
    val_idx = set(order[:n_val].tolist())
    train_raw = [item for i, item in enumerate(raw) if i not in val_idx]
    val_raw = [item for i, item in enumerate(raw) if i in val_idx]
    train_progs = get_canonical_progressions(train_raw)
    val_progs = get_canonical_progressions(val_raw)
    vocab = build_vocabulary(train_progs + val_progs)
    train_data = create_training_data(train_progs, vocab, window_size)
    val_data = create_training_data(val_progs, vocab, window_size)
    return SplitTrainingData(X_train=train_data.X, y_train=train_data.y, X_val=val_data.X, y_val=val_data.y, vocab=vocab)


Overwriting chord_dataset.py


In [6]:
%%writefile lstm_harmony_model.py
from __future__ import annotations
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path

def build_lstm_model(vocab_size: int, window_size: int = 4, embedding_dim: int = 16, lstm_units_1: int = 64, lstm_units_2: int = 32, dense_units: int = 32, dropout_rate: float = 0.3, learning_rate: float = 1e-3) -> keras.Model:
    inputs = keras.Input(shape=(window_size,), name="chord_sequence")
    x = layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True, name="chord_embedding")(inputs)
    x = layers.LSTM(units=lstm_units_1, return_sequences=True, name="lstm_layer_1")(x)
    x = layers.Dropout(dropout_rate, name="dropout_1")(x)
    x = layers.LSTM(units=lstm_units_2, return_sequences=False, name="lstm_layer_2")(x)
    x = layers.Dropout(dropout_rate, name="dropout_2")(x)
    x = layers.Dense(dense_units, activation="relu", name="dense_hidden")(x)
    x = layers.Dropout(dropout_rate * 0.67, name="dropout_3")(x)
    outputs = layers.Dense(vocab_size, activation="softmax", name="chord_probabilities")(x)
    model = keras.Model(inputs=inputs, outputs=outputs, name="RequiemLSTM")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def save_model(model: keras.Model, path: str | Path) -> None:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    keras_path = str(path / "model.keras")
    model.save(keras_path)

def load_model(path: str | Path) -> keras.Model:
    path = Path(path)
    keras_path = path / "model.keras" if path.is_dir() else path
    return keras.models.load_model(str(keras_path))

def print_model_summary(model: keras.Model) -> None:
    model.summary(expand_nested=True)


Overwriting lstm_harmony_model.py


## 3. Módulos de alto nível (motor de harmonia)

In [7]:
%%writefile requiem_model.py
from __future__ import annotations
from pathlib import Path
from typing import Optional
from tonality_adapter import DetectedNote, normalize_notes, transpose_chord, transpose_progression, normalize_progression_to_c, chord_to_roman, roman_to_chord, get_chord_pitch_classes, ALL_TONALITIES, MAJOR_TONALITIES, MINOR_TONALITIES, UT_OPTIONS, QT_OPTIONS
from audio_analyzer import detect_key, estimate_bpm
from harmony_engine import HARMONY_GRAPH, ChordResult, TwoTierMarkovModel, load_music_data, build_two_tier_markov_model, generate_progression, determine_next_chord, get_model

DEFAULT_TEMPERATURE = 0.0

class RequiemModel:
    def __init__(self, data_dir: str | Path = "music_data", tonality: str = "C", time_signature: tuple[int, int] = (4, 4), bpm: int = 120, use_neural: bool = False, neural_model_dir: str | Path = "saved_model") -> None:
        self.data_dir = Path(data_dir)
        self.tonality = tonality
        self.time_signature = time_signature
        self.bpm = bpm
        self._model: Optional[TwoTierMarkovModel] = None
        self._neural_engine = None
        self.use_neural = use_neural
        self._neural_model_dir = neural_model_dir
        self._load()
        if use_neural: self._load_neural()

    def _load(self) -> None:
        datasets = load_music_data(self.data_dir)
        self._model = build_two_tier_markov_model(datasets)

    def _load_neural(self) -> None:
        try:
            from neural_harmony_engine import NeuralHarmonyEngine
            self._neural_engine = NeuralHarmonyEngine(self._neural_model_dir)
        except Exception:
            self._neural_engine = None

    def run(self, notes: list[DetectedNote], bpm: Optional[int] = None, tonality: Optional[str] = None, time_signature: Optional[tuple[int, int]] = None, start_chord: str = "C", auto_detect: bool = True, temperature: float = DEFAULT_TEMPERATURE) -> list[ChordResult]:
        effective_tonality = tonality or self.tonality
        effective_bpm = bpm or self.bpm
        effective_ts = time_signature or self.time_signature
        if auto_detect and notes:
            if tonality is None: effective_tonality = detect_key(notes)
            if bpm is None: effective_bpm = estimate_bpm(notes)
        if self.use_neural and self._neural_engine is not None:
            return self._neural_engine.generate(notes, effective_bpm, effective_ts, start_chord, effective_tonality, False, temperature)
        local_notes = normalize_notes(notes, effective_tonality)
        raw_progression = generate_progression(local_notes, effective_bpm, effective_ts[0], effective_ts[1], start_chord, self._model)
        return [ChordResult(chord=transpose_chord(r.chord, effective_tonality), velocity=r.velocity) for r in raw_progression]

    def next_chord(self, history: list[str], notes: list[DetectedNote], window_start: float, seconds_per_beat: float, tonality: str = "C", temperature: float = DEFAULT_TEMPERATURE) -> ChordResult:
        local_notes = normalize_notes(notes, tonality)
        if self.use_neural and self._neural_engine is not None:
            result = self._neural_engine.determine_next_chord(history, local_notes, window_start, seconds_per_beat, temperature=temperature)
        else:
            result = determine_next_chord(history, local_notes, window_start, seconds_per_beat, self._model)
        return ChordResult(chord=transpose_chord(result.chord, tonality), velocity=result.velocity)

    @staticmethod
    def detect_key(notes: list[DetectedNote]) -> str: return detect_key(notes)
    @staticmethod
    def estimate_bpm(notes: list[DetectedNote]) -> int: return estimate_bpm(notes)
    @staticmethod
    def chord_pitch_classes(chord: str) -> list[int]: return get_chord_pitch_classes(chord)
    @staticmethod
    def chord_to_roman(chord: str, tonality: str) -> str: return chord_to_roman(chord, tonality)
    @staticmethod
    def roman_to_chord(roman: str, tonality: str) -> str: return roman_to_chord(roman, tonality)
    @property
    def harmony_graph(self) -> dict: return HARMONY_GRAPH
    @property
    def model(self) -> Optional[TwoTierMarkovModel]: return self._model


Overwriting requiem_model.py


In [8]:
%%writefile neural_harmony_engine.py
from __future__ import annotations
import math
from pathlib import Path
from typing import Optional
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tonality_adapter import DetectedNote, normalize_notes, transpose_chord, get_chord_pitch_classes, chord_to_roman
from harmony_engine import HARMONY_GRAPH, ChordResult, split_cinematic, WEIGHT_IN_CHORD, WEIGHT_ROOT_BONUS, WEIGHT_OUT_PENALTY, WEIGHT_STRONG_BEAT, WEIGHT_WEAK_BEAT, BEAT_MARGIN_SEC, REGISTER_LOW_THRESHOLD, REGISTER_HIGH_THRESHOLD, REGISTER_PENALTY, REGISTER_BONUS, DENSITY_THRESHOLD, ORNAMENT_PENALTY_MULTIPLIER, LEAP_THRESHOLD, TENSION_BONUS
from chord_dataset import Vocabulary, WINDOW_SIZE
from audio_analyzer import detect_key, estimate_bpm

DEFAULT_MODEL_DIR = "saved_model"
NEURAL_WEIGHT = 4.5
ACOUSTIC_WEIGHT = 1.0
DEFAULT_TEMPERATURE = 0.1
NEURAL_OVERRIDE_THRESHOLD = 0.25

def _sample_with_temperature(candidates: list[str], scores: list[float], temperature: float, rng: Optional[np.random.Generator] = None) -> str:
    rng = rng or np.random.default_rng()
    arr = np.asarray(scores, dtype=np.float64)
    arr = arr / max(temperature, 1e-6)
    arr -= arr.max()
    probs = np.exp(arr)
    probs /= probs.sum()
    idx = rng.choice(len(candidates), p=probs)
    return candidates[idx]

class NeuralHarmonyEngine:
    def __init__(self, model_dir: str | Path = DEFAULT_MODEL_DIR) -> None:
        model_dir = Path(model_dir)
        keras_file = model_dir / "model.keras"
        self._model: keras.Model = keras.models.load_model(str(keras_file))
        self._vocab = Vocabulary.load(model_dir / "vocab.json")
        self._graph_chords = list(HARMONY_GRAPH.keys())

    def _predict_probabilities(self, chord_history: list[str]) -> np.ndarray:
        context = chord_history[-WINDOW_SIZE:]
        while len(context) < WINDOW_SIZE: context = ["<PAD>"] + context
        encoded = np.array([[self._vocab.encode(c) for c in context]], dtype=np.int32)
        probs: np.ndarray = self._model.predict(encoded, verbose=0)[0]
        return probs

    def _compute_note_score(self, chord_notes: list[int], played_notes: list[DetectedNote], window_start: float, seconds_per_beat: float, is_high_density: bool, avg_amplitude: float) -> float:
        score = 0.0
        root_note = chord_notes[0]
        out_penalty = WEIGHT_OUT_PENALTY * ORNAMENT_PENALTY_MULTIPLIER if is_high_density else WEIGHT_OUT_PENALTY
        for note in played_notes:
            pc = round(note.pitch) % 12
            is_chord_tone = pc in chord_notes
            is_root = (pc == root_note)
            rel_time = note.start_time - window_start
            beat_phase = (rel_time % seconds_per_beat) / seconds_per_beat if seconds_per_beat > 0 else 0
            is_strong = beat_phase < BEAT_MARGIN_SEC or beat_phase > 1 - BEAT_MARGIN_SEC
            rhythmic_weight = WEIGHT_STRONG_BEAT if is_strong else WEIGHT_WEAK_BEAT
            amp_weight = (note.amplitude or 0.7) / avg_amplitude
            register_mod = 1.0
            if note.pitch < REGISTER_LOW_THRESHOLD and is_root: register_mod = REGISTER_BONUS
            if note.pitch > REGISTER_HIGH_THRESHOLD and not is_chord_tone: register_mod = REGISTER_PENALTY
            base_val = (WEIGHT_IN_CHORD + (WEIGHT_ROOT_BONUS if is_root else 0)) if is_chord_tone else -out_penalty
            score += base_val * rhythmic_weight * amp_weight * register_mod
        return score

    def determine_next_chord(self, chord_history: list[str], played_notes: list[DetectedNote], window_start: float, seconds_per_beat: float, temperature: float = DEFAULT_TEMPERATURE, rng: Optional[np.random.Generator] = None) -> ChordResult:
        current_chord = chord_history[-1] if chord_history else "C"
        if not played_notes: return ChordResult(chord=current_chord, velocity=0.7)
        neural_probs = self._predict_probabilities(chord_history)
        density = len(played_notes)
        is_high_density = density > DENSITY_THRESHOLD
        sum_amp = sum(n.amplitude or 0.7 for n in played_notes)
        avg_amplitude = max(0.2, sum_amp / density)
        has_tension_leap = any(abs(played_notes[i].pitch - played_notes[i-1].pitch) >= LEAP_THRESHOLD for i in range(1, density))
        current_base_roman, _ = split_cinematic(chord_to_roman(current_chord, "C"))
        current_base_name = next((c for c in HARMONY_GRAPH if chord_to_roman(c, "C") == current_base_roman), "C")
        current_node = HARMONY_GRAPH.get(current_base_name)
        best_chord = current_chord
        best_score = -math.inf
        candidates: list[str] = []
        scores: list[float] = []
        for candidate in self._graph_chords:
            cand_idx = self._vocab.encode(candidate)
            neural_prob = float(neural_probs[cand_idx]) if cand_idx > 0 else 0.0
            is_allowed_graph = (current_node is not None and candidate in current_node.allowed_transitions)
            is_allowed_neural = neural_prob > NEURAL_OVERRIDE_THRESHOLD
            if not is_allowed_graph and not is_allowed_neural: continue
            if has_tension_leap and ("7" in candidate or "dim" in candidate): neural_prob *= TENSION_BONUS
            chord_notes = get_chord_pitch_classes(candidate)
            acoustic_score_raw = self._compute_note_score(chord_notes, played_notes, window_start, seconds_per_beat, is_high_density, avg_amplitude)
            acoustic_score = acoustic_score_raw / max(1, density)
            final_score = ACOUSTIC_WEIGHT * acoustic_score + NEURAL_WEIGHT * neural_prob
            candidates.append(candidate)
            scores.append(final_score)
            if final_score > best_score:
                best_score = final_score
                best_chord = candidate
        if temperature and temperature > 0 and candidates: best_chord = _sample_with_temperature(candidates, scores, temperature, rng)
        return ChordResult(chord=best_chord, velocity=avg_amplitude)

    def generate(self, played_notes: list[DetectedNote], bpm: float = 120, time_signature: tuple[int, int] = (4, 4), start_chord: str = "C", tonality: Optional[str] = None, auto_detect: bool = True, temperature: float = DEFAULT_TEMPERATURE, seed: Optional[int] = None) -> list[ChordResult]:
        if not played_notes: return [ChordResult(chord=start_chord, velocity=0.7)]
        rng = np.random.default_rng(seed) if temperature > 0 else None
        effective_tonality = tonality or "C"
        effective_bpm = bpm
        if auto_detect:
            if tonality is None: effective_tonality = detect_key(played_notes)
            effective_bpm = estimate_bpm(played_notes) if bpm == 120 else bpm
        local_notes = normalize_notes(played_notes, effective_tonality)
        numerator, denominator = time_signature
        seconds_per_beat = (4 / denominator) * (60 / effective_bpm)
        seconds_per_measure = seconds_per_beat * numerator
        total_start = min(n.start_time for n in local_notes)
        total_end = max(n.end_time for n in local_notes)
        total_duration = total_end - total_start
        window_count = max(1, math.ceil(total_duration / seconds_per_measure))
        progression: list[ChordResult] = []
        history = [start_chord]
        for w in range(window_count):
            window_start = total_start + w * seconds_per_measure
            window_end = window_start + seconds_per_measure
            window_notes = [n for n in local_notes if n.start_time < window_end and n.end_time > window_start]
            clipped = [DetectedNote(pitch=n.pitch, start_time=max(n.start_time, window_start), end_time=min(n.end_time, window_end), amplitude=n.amplitude) for n in window_notes]
            result = self.determine_next_chord(history, clipped, window_start, seconds_per_beat, temperature=temperature, rng=rng)
            transposed_chord = transpose_chord(result.chord, effective_tonality)
            progression.append(ChordResult(chord=transposed_chord, velocity=result.velocity))
            history.append(result.chord)
        return progression

_engine_cache: Optional[NeuralHarmonyEngine] = None
def get_neural_engine(model_dir: str | Path = DEFAULT_MODEL_DIR) -> NeuralHarmonyEngine:
    global _engine_cache
    if _engine_cache is None: _engine_cache = NeuralHarmonyEngine(model_dir)
    return _engine_cache

def generate_progression_neural(played_notes: list[DetectedNote], bpm: float = 120, time_signature_numerator: int = 4, time_signature_denominator: int = 4, start_chord: str = "C", model_dir: str | Path = DEFAULT_MODEL_DIR, temperature: float = DEFAULT_TEMPERATURE) -> list[ChordResult]:
    engine = get_neural_engine(model_dir)
    return engine.generate(played_notes, bpm, (time_signature_numerator, time_signature_denominator), start_chord, auto_detect=False, temperature=temperature)


Overwriting neural_harmony_engine.py


## 4. Geração do dataset (`music_data/`)

`add_songs_to_dataset.py` cria as progressões curadas manualmente.

In [9]:
%%writefile add_songs_to_dataset.py
#!/usr/bin/env python3
import json
from pathlib import Path

DATA_DIR = Path("music_data")
DATA_DIR.mkdir(exist_ok=True)

NEW_SONGS = [
        {
        "title": "Symphony No. 5",
        "artist": "Beethoven",
        "originalTonality": "Cm",
        "normalizedProgression": ["Cm", "G", "Cm", "Ab", "Eb", "Bb", "Cm", "G7", "Cm"],
    },
    {
        "title": "Canon in D",
        "artist": "Pachelbel",
        "originalTonality": "D",
        "normalizedProgression": ["D", "A", "Bm", "F#m", "G", "D", "G", "A"],
    },
    {
        "title": "Eine Kleine Nachtmusik",
        "artist": "Mozart",
        "originalTonality": "G",
        "normalizedProgression": ["G", "D", "G", "D7", "G", "C", "G", "D", "G"],
    },
    {
        "title": "Ode to Joy",
        "artist": "Beethoven",
        "originalTonality": "D",
        "normalizedProgression": ["D", "A", "D", "G", "D", "A", "D", "A", "D"],
    },
    {
        "title": "Lacrimosa",
        "artist": "Mozart",
        "originalTonality": "Dm",
        "normalizedProgression": ["Dm", "A7", "Dm", "Gm", "A7", "Dm", "Bb", "A7", "Dm"],
    },
    {
        "title": "Toccata and Fugue",
        "artist": "Bach",
        "originalTonality": "Dm",
        "normalizedProgression": ["Dm", "A7", "Dm", "C", "Bb", "A", "Dm", "Gm", "A7", "Dm"],
    },
    {
        "title": "Gymnopédie No. 1",
        "artist": "Satie",
        "originalTonality": "G",
        "normalizedProgression": ["Gmaj7", "Cmaj7", "Gmaj7", "Cmaj7", "Amaj7", "Dmaj7"],
    },
    {
        "title": "Prelude BWV 846",
        "artist": "Bach",
        "originalTonality": "C",
        "normalizedProgression": ["C", "Dm", "G7", "C", "Am", "D7", "G", "C"],
    },
    {
        "title": "Waltz Op. 64 No. 2",
        "artist": "Chopin",
        "originalTonality": "C#m",
        "normalizedProgression": ["C#m", "G#7", "C#m", "F#m", "B7", "E", "A", "G#7", "C#m"],
    },
    {
        "title": "Nocturne Op. 9 No. 2",
        "artist": "Chopin",
        "originalTonality": "Eb",
        "normalizedProgression": ["Eb", "Cm", "Ab", "Bb", "Eb", "Bb7", "Eb", "Gm", "Ab", "Bb7", "Eb"],
    },
    {
        "title": "Back in Black",
        "artist": "AC/DC",
        "originalTonality": "Em",
        "normalizedProgression": ["E", "D", "A", "E", "D", "A", "E"],
    },
    {
        "title": "Paranoid",
        "artist": "Black Sabbath",
        "originalTonality": "Em",
        "normalizedProgression": ["Em", "D", "C", "D", "Em", "G", "D", "Em"],
    },
    {
        "title": "Whole Lotta Love",
        "artist": "Led Zeppelin",
        "originalTonality": "E",
        "normalizedProgression": ["E", "D", "E", "D", "A", "E"],
    },
    {
        "title": "Highway to Hell",
        "artist": "AC/DC",
        "originalTonality": "A",
        "normalizedProgression": ["A", "D", "G", "D", "A", "D", "G", "D", "A"],
    },
    {
        "title": "Stairway to Heaven Solo",
        "artist": "Led Zeppelin",
        "originalTonality": "Am",
        "normalizedProgression": ["Am", "G", "F", "G", "Am", "G", "F", "E"],
    },
    {
        "title": "Welcome to the Black Parade",
        "artist": "My Chemical Romance",
        "originalTonality": "G",
        "normalizedProgression": ["G", "D", "Em", "Bm", "C", "G", "Am", "D"],
    },
    {
        "title": "Angel of Death",
        "artist": "Slayer",
        "originalTonality": "Cm",
        "normalizedProgression": ["Cm", "Bb", "Ab", "G", "Cm", "Eb", "Bb", "G7", "Cm"],
    },
    {
        "title": "Uptown Funk",
        "artist": "Mark Ronson ft. Bruno Mars",
        "originalTonality": "Dm",
        "normalizedProgression": ["Dm", "G", "Dm", "G", "Dm", "A7", "Dm"],
    },
    {
        "title": "Bad Guy",
        "artist": "Billie Eilish",
        "originalTonality": "Gm",
        "normalizedProgression": ["Gm", "Dm", "Eb", "Cm", "Gm", "Dm", "Eb", "Cm"],
    },
    {
        "title": "Drivers License",
        "artist": "Olivia Rodrigo",
        "originalTonality": "Db",
        "normalizedProgression": ["Db", "Bbm", "Gb", "Ab", "Db", "Bbm", "Gb", "Ab7"],
    },
    {
        "title": "Levitating",
        "artist": "Dua Lipa",
        "originalTonality": "Bm",
        "normalizedProgression": ["Bm", "D", "F#m", "G", "Bm", "D", "Em", "A"],
    },
    {
        "title": "As It Was",
        "artist": "Harry Styles",
        "originalTonality": "Am",
        "normalizedProgression": ["Am", "C", "G", "Fmaj7", "Am", "C", "G", "F"],
    },
    {
        "title": "Flowers",
        "artist": "Miley Cyrus",
        "originalTonality": "Ab",
        "normalizedProgression": ["Ab", "Fm", "Eb", "Db", "Ab", "Fm", "Eb", "Bb"],
    },
    {
        "title": "Heat Waves",
        "artist": "Glass Animals",
        "originalTonality": "F",
        "normalizedProgression": ["F", "Am", "C", "G", "F", "Am", "Bb", "C"],
    },
    {
        "title": "Watermelon Sugar",
        "artist": "Harry Styles",
        "originalTonality": "G",
        "normalizedProgression": ["G", "Am", "C", "D", "G", "Am", "C", "D"],
    },
    {
        "title": "Isn't She Lovely",
        "artist": "Stevie Wonder",
        "originalTonality": "Eb",
        "normalizedProgression": ["Eb", "Ab", "Eb", "Fm7", "Bb7", "Eb"],
    },
    {
        "title": "Superstition",
        "artist": "Stevie Wonder",
        "originalTonality": "Eb",
        "normalizedProgression": ["Eb7", "Ab7", "Eb7", "Bb7", "Ab7", "Eb7"],
    },
    {
        "title": "What a Wonderful World",
        "artist": "Louis Armstrong",
        "originalTonality": "F",
        "normalizedProgression": ["F", "Am", "Bb", "Am", "Gm", "Am", "Bb", "C7", "F"],
    },
    {
        "title": "Fly Me to the Moon",
        "artist": "Frank Sinatra",
        "originalTonality": "C",
        "normalizedProgression": ["Am", "Dm", "G7", "C", "Am", "Dm", "G7", "Cmaj7"],
    },
    {
        "title": "Autumn Leaves",
        "artist": "Jazz Standard",
        "originalTonality": "Gm",
        "normalizedProgression": ["Cm", "F7", "Bb", "Eb", "Am7", "D7", "Gm", "Gm"],
    },
    {
        "title": "Blue Bossa",
        "artist": "Kenny Dorham",
        "originalTonality": "Cm",
        "normalizedProgression": ["Cm", "Fm", "Dm7", "G7", "Cm", "Ab", "Db", "G7", "Cm"],
    },
    {
        "title": "Sandstorm",
        "artist": "Darude",
        "originalTonality": "Bm",
        "normalizedProgression": ["Bm", "A", "G", "F#m", "Bm", "A", "G", "A"],
    },
    {
        "title": "Infinity",
        "artist": "Guru Josh",
        "originalTonality": "Am",
        "normalizedProgression": ["Am", "F", "C", "G", "Am", "F", "G", "E"],
    },
    {
        "title": "Strobe",
        "artist": "deadmau5",
        "originalTonality": "Am",
        "normalizedProgression": ["Am", "C", "G", "Dm", "Am", "F", "C", "Em"],
    },
    {
        "title": "Comptine d Un Autre Ete",
        "artist": "Yann Tiersen",
        "originalTonality": "Dm",
        "normalizedProgression": ["Dm", "Am", "F", "C", "G", "Bb", "F", "C", "Dm"],
    },
    {
        "title": "Experience",
        "artist": "Ludovico Einaudi",
        "originalTonality": "D",
        "normalizedProgression": ["D", "Bm", "G", "A", "D", "F#m", "G", "A"],
    },
    {
        "title": "Nuvole Bianche",
        "artist": "Ludovico Einaudi",
        "originalTonality": "E",
        "normalizedProgression": ["E", "B", "C#m", "A", "E", "B", "G#m", "A"],
    },
    {
        "title": "Married Life",
        "artist": "Michael Giacchino - Up",
        "originalTonality": "C",
        "normalizedProgression": ["C", "G7", "Am", "Dm", "G7", "C", "E7", "Am", "F", "G7", "C"],
    },
    {
        "title": "Hans Zimmer Test of Time",
        "artist": "Hans Zimmer",
        "originalTonality": "Am",
        "normalizedProgression": ["Am", "G", "F", "E", "Am", "G", "F", "G", "Am"],
    },
    {
        "title": "How Great Thou Art",
        "artist": "Traditional",
        "originalTonality": "G",
        "normalizedProgression": ["G", "C", "G", "D", "G", "C", "G", "D7", "G"],
    },
    {
        "title": "10000 Reasons",
        "artist": "Matt Redman",
        "originalTonality": "G",
        "normalizedProgression": ["G", "D", "Em", "C", "G", "D", "Em", "C", "G"],
    },
    {
        "title": "Wave",
        "artist": "Tom Jobim",
        "originalTonality": "D",
        "normalizedProgression": ["Dmaj7", "G7", "Cmaj7", "F7", "Bm7", "E7", "Am7", "D7", "Gmaj7"],
    },
    {
        "title": "O Pato",
        "artist": "Jaime Silva",
        "originalTonality": "C",
        "normalizedProgression": ["C", "G7", "Am", "D7", "G7", "C"],
    },
    {
        "title": "Besame Mucho",
        "artist": "Consuelo Velázquez",
        "originalTonality": "Dm",
        "normalizedProgression": ["Dm", "A7", "Dm", "Gm", "Dm", "A7", "Dm", "E7", "A7", "Dm"],
    },
    {
        "title": "Creep",
        "artist": "Radiohead",
        "originalTonality": "G",
        "normalizedProgression": ["G", "B", "C", "Cm", "G", "B", "C", "Cm"],
    },
    {
        "title": "Karma Police",
        "artist": "Radiohead",
        "originalTonality": "Am",
        "normalizedProgression": ["Am", "E7", "Am", "E7", "C", "G", "D", "Am"],
    },
    {
        "title": "Mr Brightside",
        "artist": "The Killers",
        "originalTonality": "Bb",
        "normalizedProgression": ["Bb", "F", "Gm", "Eb", "Bb", "F", "Gm", "Eb"],
    },
    {
        "title": "Seven Nation Army",
        "artist": "The White Stripes",
        "originalTonality": "Em",
        "normalizedProgression": ["Em", "G", "Em", "D", "C", "B7", "Em"],
    },
    {
        "title": "Come As You Are",
        "artist": "Nirvana",
        "originalTonality": "Dm",
        "normalizedProgression": ["Dm", "Am", "Dm", "Am", "C", "G", "Dm"],
    },
    {
        "title": "Lithium",
        "artist": "Nirvana",
        "originalTonality": "D",
        "normalizedProgression": ["D", "F#", "Bm", "G", "Bb", "C", "A", "D"],
    },
    {
        "title": "Take Me Home Country Roads",
        "artist": "John Denver",
        "originalTonality": "G",
        "normalizedProgression": ["G", "D", "Em", "C", "G", "D", "G"],
    },
    {
        "title": "Friends in Low Places",
        "artist": "Garth Brooks",
        "originalTonality": "A",
        "normalizedProgression": ["A", "Bm", "E7", "A", "Bm", "E7", "A"],
    },
    {
        "title": "Autumn Leaves (Jazz Reharmonized)",
        "originalTonality": "Am",
        "normalizedProgression": ["Cm7", "F7", "Bbmaj7", "Ebmaj7", "Am7b5", "D7", "Gm", "Gm", "Cm7", "F7", "Bbmaj7", "Ebmaj7", "Am7b5", "D7", "Gm7", "Gm7"],
    },
    {
        "title": "Summertime",
        "originalTonality": "Am",
        "normalizedProgression": ["Am", "E7", "Am", "Am", "Dm", "Am", "E7", "Am", "Am", "E7", "Am", "C", "Dm", "F", "E7", "Am"],
    },
    {
        "title": "Misty",
        "originalTonality": "F",
        "normalizedProgression": ["Fmaj7", "Cm7", "F7", "Bbmaj7", "Bbm7", "Eb7", "Fmaj7", "Dm7", "Gm7", "C7", "Am7", "D7", "Gm7", "C7"],
    },
    {
        "title": "Satin Doll",
        "originalTonality": "C",
        "normalizedProgression": ["Dm7", "G7", "Dm7", "G7", "Em7", "A7", "Em7", "A7", "Am7", "D7", "Abm7", "Db7", "Cmaj7", "Gm7", "C7"],
    },
    {
        "title": "Round Midnight",
        "originalTonality": "Cm",
        "normalizedProgression": ["Cm", "Ebdim", "Dm7b5", "G7b9", "Cm", "Cm7", "Fm", "Dm7b5", "G7", "Cm", "Abmaj7", "G7", "Cm"],
    },
    {
        "title": "Bye Bye Blackbird",
        "originalTonality": "F",
        "normalizedProgression": ["F", "Fm", "C", "A7", "Dm", "G7", "Gm7", "C7", "F", "Fm", "C", "A7", "Dm", "G7", "C7", "F"],
    },
    {
        "title": "There Will Never Be Another You",
        "originalTonality": "Eb",
        "normalizedProgression": ["Ebmaj7", "Ab7", "Gm7", "C7", "Fm7", "Bb7", "Ebmaj7", "Cm7", "Fm7", "Bb7", "Ebmaj7", "Gm7", "C7", "Fm7", "Bb7", "Ebmaj7"],
    },
    {
        "title": "Desafinado",
        "originalTonality": "F",
        "normalizedProgression": ["Fmaj7", "G7", "Gm7", "C7", "Fmaj7", "F7", "Bbmaj7", "Bb6", "Am7", "D7", "Gm7", "C7", "Am7", "D7", "Gm7", "C7", "Fmaj7"],
    },
    {
        "title": "Corcovado",
        "originalTonality": "D",
        "normalizedProgression": ["Dm7", "G7", "Cmaj7", "C6", "Cm7", "F7", "Bbmaj7", "E7", "A7", "D", "Em7", "A7", "D", "D7", "G", "Gm", "D"],
    },
    {
        "title": "Mas Que Nada",
        "originalTonality": "Bm",
        "normalizedProgression": ["Bm", "Em", "Bm", "Em", "Bm", "F#7", "Bm", "Bm", "Em", "Bm", "Em", "Bm", "F#7", "Bm"],
    },
    {
        "title": "Triste",
        "originalTonality": "Bb",
        "normalizedProgression": ["Bbmaj7", "Bbm7", "Eb7", "Abmaj7", "Dm7", "G7", "Cm7", "F7", "Bbmaj7", "Bbm7", "Eb7", "Abmaj7", "Am7", "D7", "Gm7", "C7", "Cm7", "F7", "Bb"],
    },
    {
        "title": "Chega de Saudade",
        "originalTonality": "D",
        "normalizedProgression": ["Dm", "E7", "Am", "Dm", "E7", "Am", "Gm", "A7", "Dm", "C#7", "F#", "Bm", "E7", "A7", "D", "A7", "D"],
    },
    {
        "title": "Paranoid Android",
        "originalTonality": "Am",
        "normalizedProgression": ["Am", "G", "F", "Em", "Am", "G", "F", "Em", "C", "G", "Em", "C", "G", "D", "Am", "E7"],
    },
    {
        "title": "Under the Bridge",
        "originalTonality": "E",
        "normalizedProgression": ["E", "B", "C#m", "G#m", "A", "E", "B", "C#m", "G#m", "A", "E", "B", "F#m", "E", "B", "C#m", "A", "E", "B"],
    },
    {
        "title": "Wish You Were Here (verse)",
        "originalTonality": "G",
        "normalizedProgression": ["Em", "G", "Em", "G", "Em", "A", "Em", "A", "G", "D", "Am", "G", "D", "Am", "G"],
    },
    {
        "title": "Black",
        "originalTonality": "E",
        "normalizedProgression": ["E", "A", "E", "A", "E", "A", "B", "C#m", "A", "B", "E", "A", "E", "A"],
    },
    {
        "title": "Fade to Black",
        "originalTonality": "Bm",
        "normalizedProgression": ["Bm", "G", "Bm", "G", "Bm", "A", "E", "Bm", "Bm", "G", "A", "Bm", "G", "A", "E", "Bm"],
    },
    {
        "title": "Welcome Home (Coheed)",
        "originalTonality": "Am",
        "normalizedProgression": ["Am", "G", "F", "C", "Am", "G", "F", "G", "Am", "Em", "F", "G", "C", "G", "Am", "F"],
    },
    {
        "title": "Concerning Hobbits",
        "originalTonality": "D",
        "normalizedProgression": ["D", "G", "D", "G", "D", "A", "G", "D", "Bm", "G", "A", "D", "G", "D", "A", "D"],
    },
    {
        "title": "Requiem for a Dream",
        "originalTonality": "Dm",
        "normalizedProgression": ["Dm", "A7", "Dm", "Gm", "A7", "Dm", "Dm", "Gm", "Dm", "A7", "Dm"],
    },
    {
        "title": "The Last of Us Theme",
        "originalTonality": "Am",
        "normalizedProgression": ["Am", "C", "G", "Am", "F", "C", "G", "Am", "Em", "F", "C", "G", "Am", "F", "G", "Am"],
    },
    {
        "title": "Ori and the Blind Forest",
        "originalTonality": "C",
        "normalizedProgression": ["C", "Am", "F", "G", "C", "Em", "F", "G", "Am", "F", "C", "G", "Am", "F", "G", "C"],
    },
    {
        "title": "Skyrim Main Theme (Dragonborn)",
        "originalTonality": "Dm",
        "normalizedProgression": ["Dm", "C", "Dm", "C", "Bb", "C", "Dm", "Dm", "Gm", "Dm", "A7", "Dm", "Bb", "C", "Dm"],
    },
    {
        "title": "I Will Always Love You",
        "originalTonality": "G",
        "normalizedProgression": ["G", "Bm", "C", "D", "G", "Bm", "C", "D", "G", "Em", "C", "D", "G", "Am", "D", "G"],
    },
    {
        "title": "Superstition (full)",
        "originalTonality": "Cm",
        "normalizedProgression": ["Cm", "Cm", "Cm", "Cm", "Eb", "Eb", "Cm", "Cm", "Fm", "Fm", "Cm", "Cm", "G7", "F7", "Cm", "Cm"],
    },
    {
        "title": "Ain't No Sunshine",
        "originalTonality": "Am",
        "normalizedProgression": ["Am", "Am", "Am", "Em", "G7", "Am", "Am", "Am", "Dm", "Dm", "Am", "Am", "G7", "Am"],
    },
    {
        "title": "What's Going On",
        "originalTonality": "E",
        "normalizedProgression": ["Emaj7", "A6", "Emaj7", "A6", "C#m7", "F#m7", "C#m7", "F#m7", "Emaj7", "A6", "Emaj7", "A6"],
    },
    {
        "title": "Dancing Queen",
        "originalTonality": "A",
        "normalizedProgression": ["A", "E", "D", "A", "A", "E", "D", "A", "Bm", "E7", "A", "D", "A", "Bm", "E7", "A"],
    },
    {
        "title": "Africa (Toto)",
        "originalTonality": "F#m",
        "normalizedProgression": ["F#m", "D", "A", "E", "F#m", "D", "A", "E", "Bm", "E", "A", "D", "Bm", "E", "A", "F#m"],
    },
    {
        "title": "Mr. Brightside (chorus)",
        "originalTonality": "Bb",
        "normalizedProgression": ["Bb", "F", "Gm", "Eb", "Bb", "F", "Gm", "Eb", "Bb", "F", "Gm", "Eb"],
    },
    {
        "title": "Somebody That I Used to Know",
        "originalTonality": "Dm",
        "normalizedProgression": ["Dm", "C", "Bb", "Dm", "C", "Bb", "Dm", "C", "Bb", "A7", "Dm", "C", "Bb", "F", "C", "Dm"],
    },
    {
        "title": "Shape of My Heart (Sting)",
        "originalTonality": "F#m",
        "normalizedProgression": ["F#m", "C#m", "D", "A", "F#m", "C#m", "D", "E", "F#m", "D", "A", "E", "F#m", "D", "A", "E", "F#m"],
    },
    {
        "title": "Prelude Op 28 No 4 Chopin",
        "originalTonality": "Em",
        "normalizedProgression": ["Em", "Em", "B7", "B7", "Em", "C", "G", "Am", "B7", "Em", "Fm", "C", "G", "B7", "Em"],
    },
    {
        "title": "Sonatina in G Major (Clementi)",
        "originalTonality": "G",
        "normalizedProgression": ["G", "D7", "G", "D", "G", "D7", "G", "C", "G", "D7", "G", "Em", "A7", "D", "G", "D7", "G"],
    },
    {
        "title": "Für Elise (extended)",
        "originalTonality": "Am",
        "normalizedProgression": ["Am", "E7", "Am", "E7", "Am", "C", "G", "Am", "E7", "Am", "F", "C", "G", "Am", "E7", "Am"],
    },
    {
        "title": "Waltz Op 69 No 1 Chopin",
        "originalTonality": "Ab",
        "normalizedProgression": ["Ab", "Eb7", "Ab", "Db", "Ab", "Eb7", "Ab", "Fm", "Bbm", "Eb7", "Ab", "Bbm", "Eb7", "Ab"],
    },
    {
        "title": "Symphony No. 5", "artist": "Beethoven",
        "originalTonality": "Cm",
        "normalizedProgression": ["Cm", "G", "Cm", "Ab", "Eb", "Bb", "Cm", "G7", "Cm"]
    },
    {
        "title": "Canon in D", "artist": "Pachelbel",
        "originalTonality": "D",
        "normalizedProgression": ["D", "A", "Bm", "F#m", "G", "D", "G", "A"]
    },
    {
        "title": "Eine Kleine Nachtmusik", "artist": "Mozart",
        "originalTonality": "G", "normalizedProgression": ["G", "D", "G", "D7", "G", "C", "G", "D", "G"]
    },
    {
        "title": "Creep", "artist": "Radiohead",
        "originalTonality": "G",
        "normalizedProgression": ["G", "B", "C", "Cm", "G", "B", "C", "Cm"]
    },
    {
        "title": "Karma Police", "artist": "Radiohead",
        "originalTonality": "Am",
        "normalizedProgression": ["Am", "E7", "Am", "E7", "C", "G", "D", "Am"]
    },
    {
        "title": "Seven Nation Army", "artist": "The White Stripes",
        "originalTonality": "Em",
        "normalizedProgression": ["Em", "G", "Em", "D", "C", "B7", "Em"]
    },
    {
        "title": "Come As You Are", "artist": "Nirvana",
        "originalTonality": "Dm",
        "normalizedProgression": ["Dm", "Am", "Dm", "Am", "C", "G", "Dm"]
    },
    {
        "title": "Take Me Home Country Roads", "artist": "John Denver",
        "originalTonality": "G",
        "normalizedProgression": ["G", "D", "Em", "C", "G", "D", "G"]
    }
]

def slugify_title(title: str) -> str:
    name = title.lower()
    for ch in " ()-,.'!?/": name = name.replace(ch, "_")
    return name.strip("_") + ".json"

def save_song(song: dict, overwrite: bool = False) -> bool:
    filename = slugify_title(song["title"])
    path = DATA_DIR / filename
    if path.exists() and not overwrite: return False
    path.write_text(json.dumps(song, ensure_ascii=False, indent=2), encoding="utf-8")
    return True

def main() -> None:
    added = sum(1 for song in NEW_SONGS if save_song(song))
    print(f"Adicionado {added} músicas.")

if __name__ == "__main__":
    main()


Overwriting add_songs_to_dataset.py


In [10]:
!python3 add_songs_to_dataset.py

Adicionado 0 músicas.


## 5. Treinamento do modelo — **gera os pesos**

In [11]:
%%writefile train_model.py
from __future__ import annotations
import json
import os
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras
from chord_dataset import prepare_train_val_split, WINDOW_SIZE
from lstm_harmony_model import build_lstm_model, save_model

DATA_DIR = "music_data"
MODEL_DIR = "saved_model"
HISTORY_FILE = "training_history.json"
BATCH_SIZE = 32
MAX_EPOCHS = 300
PATIENCE = 25
VALIDATION_SPLIT = 0.2
LEARNING_RATE = 1e-3
SEED = 42

def train():
    tf.random.set_seed(SEED)
    np.random.seed(SEED)
    data = prepare_train_val_split(DATA_DIR, WINDOW_SIZE, VALIDATION_SPLIT, SEED)
    Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
    data.vocab.save(Path(MODEL_DIR) / "vocab.json")
    indices = np.random.permutation(len(data.X_train))
    X_shuffled = data.X_train[indices]
    y_shuffled = data.y_train[indices]
    vocab_size = len(data.vocab)
    model = build_lstm_model(vocab_size=vocab_size, window_size=WINDOW_SIZE, embedding_dim=16, lstm_units_1=64, lstm_units_2=32, dense_units=32, dropout_rate=0.3, learning_rate=LEARNING_RATE)
    early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1)
    checkpoint = keras.callbacks.ModelCheckpoint(filepath=str(Path(MODEL_DIR) / "checkpoint.keras"), monitor="val_loss", save_best_only=True, verbose=0)
    reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10, min_lr=1e-6, verbose=1)
    history = model.fit(X_shuffled, y_shuffled, batch_size=BATCH_SIZE, epochs=MAX_EPOCHS, validation_data=(data.X_val, data.y_val), callbacks=[early_stopping, checkpoint, reduce_lr], verbose=1)
    save_model(model, MODEL_DIR)
    history_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    Path(HISTORY_FILE).write_text(json.dumps(history_data, indent=2))
    print("\n✓ Treinamento concluído e modelo salvo!")

if __name__ == "__main__":
    train()


Overwriting train_model.py


In [12]:
!python3 train_model.py

2026-09-23 01:44:05.464542: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-23 01:44:05.582785: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-23 01:44:16.213578: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
Epoch 1/300
25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 43ms/step - accuracy: 0.1541 - loss: 3.8587 - val_accuracy: 0.1633 - val_loss: 3.7764 - learning_rate: 0.0010
Epoch 2/300
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1771 - loss: 3.5229 - val_accuracy: 0.1633 - val_loss: 3.0204 - learning_rate: 0.0010
Epoch 3/300
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step -

## 6. Exportação dos pesos

In [13]:
!mkdir -p ../requiem-app-source/requiem-app/public/model

In [14]:
%%writefile export_weights_json.py
import os
import json
import base64
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import tensorflow as tf
from tensorflow import keras

def main():
    keras_path = Path("saved_model/model.keras")
    output_dir = Path("../requiem-app-source/requiem-app/public/model")
    output_dir.mkdir(parents=True, exist_ok=True)
    model = keras.models.load_model(str(keras_path))
    weights_data = {}
    for layer in model.layers:
        layer_weights = layer.get_weights()
        if not layer_weights: continue
        weights_data[layer.name] = {"class": layer.__class__.__name__, "weights": []}
        for w in layer_weights:
            w_np = np.array(w, dtype=np.float32)
            weights_data[layer.name]["weights"].append({"shape": list(w_np.shape), "dtype": "float32", "data_b64": base64.b64encode(w_np.tobytes()).decode("ascii")})
    output = {"model_name": "RequiemLSTM", "vocab_size": 97, "window_size": 4, "embedding_dim": 32, "lstm_units": [128, 64], "dense_units": [64], "total_params": model.count_params(), "layers": weights_data}
    out_path = output_dir / "weights.json"
    out_path.write_text(json.dumps(output, separators=(",", ":")))
    import shutil
    if Path("saved_model/vocab.json").exists(): shutil.copy2(Path("saved_model/vocab.json"), output_dir / "vocab.json")

if __name__ == "__main__":
    main()


Overwriting export_weights_json.py


In [15]:
!python3 export_weights_json.py

## 7. Exportando todos os arquivos necessários

In [16]:
%%writefile exportar_tudo.py
import os
import json
import base64
import shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras
from pathlib import Path

# Cria a pasta de destino
out_dir = Path("export_frontend")
out_dir.mkdir(exist_ok=True)

print("A carregar o modelo...")
model = keras.models.load_model("saved_model/model.keras")

# 1. Exportar weights.json (Usado pelo seu novo motor TS manual)
print("A gerar weights.json...")
weights_data = {}
for layer in model.layers:
    lw = layer.get_weights()
    if not lw: continue
    weights_data[layer.name] = {"class": layer.__class__.__name__, "weights": []}
    for w in lw:
        w_np = np.array(w, dtype=np.float32)
        weights_data[layer.name]["weights"].append({
            "shape": list(w_np.shape),
            "dtype": "float32",
            "data_b64": base64.b64encode(w_np.tobytes()).decode("ascii")
        })

output_json = {
    "model_name": "RequiemLSTM",
    "total_params": model.count_params(),
    "layers": weights_data
}
(out_dir / "weights.json").write_text(json.dumps(output_json, separators=(",", ":")))

# 2. Exportar model.json e group1-shard1of1.bin (Formato TF.js Clássico)
print("A gerar model.json e group1-shard1of1.bin...")
weight_manifests = []
all_weights_bytes = bytearray()
for layer in model.layers:
    lw = layer.get_weights()
    if not lw: continue
    layer_weight_names = [f"{layer.name}/{w.name}" for w in layer.weights]
    for weight_array, weight_name in zip(lw, layer_weight_names):
        weight_np = np.array(weight_array, dtype=np.float32)
        weight_bytes = weight_np.tobytes()
        all_weights_bytes.extend(weight_bytes)
        weight_manifests.append({
            "name": weight_name,
            "shape": list(weight_np.shape),
            "dtype": "float32",
            "quantization": None,
        })

bin_filename = "group1-shard1of1.bin"
(out_dir / bin_filename).write_bytes(bytes(all_weights_bytes))

model_json = {
    "format": "layers-model",
    "generatedBy": "keras",
    "convertedBy": "manual_script",
    "modelTopology": {
        "class_name": model.__class__.__name__,
        "config": model.get_config(),
        "keras_version": keras.__version__,
        "backend": "tensorflow",
    },
    "weightsManifest": [{"paths": [bin_filename], "weights": weight_manifests}],
}
(out_dir / "model.json").write_text(json.dumps(model_json, indent=2))

# 3. Copiar vocab.json
print("A copiar vocab.json...")
shutil.copy2("saved_model/vocab.json", out_dir / "vocab.json")

print(f"\nSucesso! Os 4 ficheiros foram criados na pasta: {out_dir.absolute()}")

Overwriting exportar_tudo.py


In [17]:
!python3 exportar_tudo.py

2026-09-23 01:45:04.554325: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-23 01:45:04.624894: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
A carregar o modelo...
2026-09-23 01:45:09.914449: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
A gerar weights.json...
A gerar model.json e group1-shard1of1.bin...
A copiar vocab.json...

Sucesso! Os 4 ficheiros foram criados na pasta: /content/export_frontend
